# Student Lifestyle and Stress Level Prediction

**Goal:** predict whether a student is stressed (`Stress_Level` = 1) or not (0) from lifestyle and academic factors, and find out which factors matter most.

**Dataset:** 25,500 students with sleep, study and social media hours, attendance, exam pressure, family support, month and student type (school / college / working student).

**Workflow**
1. Load and inspect the data
2. Clean it (missing values, impossible values)
3. Exploratory data analysis
4. Train and compare classification models
5. Evaluate, interpret the important features, and conclude

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             RocCurveDisplay, classification_report)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42
os.makedirs("figures", exist_ok=True)

def save(fig, name):
    fig.savefig(f"figures/{name}.png", dpi=150, bbox_inches="tight")

## 1. Load and inspect the data

In [ ]:
df = pd.read_csv("student-lifestyle-and-stress-dataset.csv")
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T.round(2)

## 2. Data quality and cleaning

Real-world data is rarely clean. Two problems show up here: **missing values** (about 5% in every column except the target) and **impossible values** (negative study hours, attendance above 100%, and so on).

In [ ]:
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "percent": (df.isna().mean() * 100).round(2),
})
missing

In [ ]:
# Rules for what counts as a valid value
invalid = {
    "Study_Hours < 0":            (df["Study_Hours"] < 0).sum(),
    "Study_Hours > 16":           (df["Study_Hours"] > 16).sum(),
    "Attendance <= 0":            (df["Attendance"] <= 0).sum(),
    "Attendance > 100":           (df["Attendance"] > 100).sum(),
    "Sleep_Hours outside 0-24":   ((df["Sleep_Hours"] < 0) | (df["Sleep_Hours"] > 24)).sum(),
    "Social_Media_Hours < 0":     (df["Social_Media_Hours"] < 0).sum(),
}
pd.Series(invalid, name="rows").to_frame()

In [ ]:
clean = df.copy()

# Impossible values become missing; they are filled in later together with the real gaps.
clean.loc[(clean["Study_Hours"] < 0) | (clean["Study_Hours"] > 16), "Study_Hours"] = np.nan
# Attendance of exactly 0 or -5 is a pile-up at the edge (looks like clipped bad entries), so it is treated as invalid too.
clean.loc[(clean["Attendance"] <= 0) | (clean["Attendance"] > 100), "Attendance"] = np.nan

# Exact duplicate rows would leak between train and test, so drop them
print("Duplicate rows removed:", clean.duplicated().sum())
clean = clean.drop_duplicates().reset_index(drop=True)
print("Rows remaining:", len(clean))

print("Missing values after cleaning rules:")
print(clean.isna().sum())

**Why not fill the gaps right now?** If we fill missing values using statistics from the *whole* dataset, information from the test set leaks into training. So imputation (median for numbers, most frequent for `Student_Type`) is done inside the modeling pipeline in section 4, fitted on the training data only.

## 3. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

counts = clean["Stress_Level"].value_counts().sort_index()
axes[0].bar(["Not stressed (0)", "Stressed (1)"], counts.values, color=["#4C78A8", "#E45756"])
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f"{v:,}\n({v / counts.sum():.1%})", ha="center")
axes[0].set_title("Target balance")
axes[0].set_ylabel("Students")
axes[0].set_ylim(0, counts.max() * 1.15)

rate = clean.groupby("Student_Type")["Stress_Level"].mean().sort_values(ascending=False) * 100
sns.barplot(x=rate.index, y=rate.values, ax=axes[1], color="#E45756")
for i, v in enumerate(rate.values):
    axes[1].text(i, v + 0.5, f"{v:.1f}%", ha="center")
axes[1].set_title("Stress rate by student type")
axes[1].set_ylabel("% stressed")
axes[1].set_xlabel("")
axes[1].set_ylim(0, rate.max() * 1.2)

plt.tight_layout()
save(fig, "01_target_and_student_type")
plt.show()

In [ ]:
num_cols = ["Sleep_Hours", "Study_Hours", "Social_Media_Hours", "Attendance", "Exam_Pressure", "Family_Support"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), num_cols):
    sns.kdeplot(data=clean, x=col, hue="Stress_Level", common_norm=False, fill=True,
                palette={0: "#4C78A8", 1: "#E45756"}, ax=ax, legend=False)
    ax.set_title(col.replace("_", " "))
    ax.set_xlabel("")
fig.legend(["Stressed (1)", "Not stressed (0)"], loc="upper center", ncol=2, frameon=False)
plt.tight_layout(rect=[0, 0, 1, 0.95])
save(fig, "02_feature_distributions_by_stress")
plt.show()

In [ ]:
corr = clean[num_cols + ["Month", "Stress_Level"]].corr()

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True, ax=ax)
ax.set_title("Correlation matrix")
plt.tight_layout()
save(fig, "03_correlation_heatmap")
plt.show()

corr["Stress_Level"].drop("Stress_Level").sort_values(key=abs, ascending=False).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

ep = clean.groupby("Exam_Pressure")["Stress_Level"].mean() * 100
axes[0].plot(ep.index, ep.values, marker="o", color="#E45756")
axes[0].set_title("Stress rate by exam pressure")
axes[0].set_xlabel("Exam pressure (1-10)")
axes[0].set_ylabel("% stressed")

mo = clean.groupby("Month")["Stress_Level"].mean() * 100
axes[1].bar(mo.index.astype(int), mo.values, color="#4C78A8")
axes[1].set_title("Stress rate by month")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("% stressed")
axes[1].set_xticks(range(1, 13))

plt.tight_layout()
save(fig, "04_stress_by_exam_pressure_and_month")
plt.show()

In [ ]:
# Study hours and sleep, binned, to see the shape of the relationship
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, col, bins in zip(axes,
                         ["Study_Hours", "Sleep_Hours", "Social_Media_Hours"],
                         [8, 8, 8]):
    b = pd.cut(clean[col], bins=bins)
    r = clean.groupby(b, observed=True)["Stress_Level"].mean() * 100
    ax.bar(range(len(r)), r.values, color="#59A14F")
    ax.set_xticks(range(len(r)))
    ax.set_xticklabels([f"{i.left:.1f}-{i.right:.1f}" for i in r.index], rotation=45, ha="right")
    ax.set_title(f"Stress rate by {col.replace('_', ' ').lower()}")
    ax.set_ylabel("% stressed")
plt.tight_layout()
save(fig, "05_stress_by_hours")
plt.show()

## 4. Modeling

**Setup**
- 80/20 train/test split, stratified so both sets keep the 30% stressed share.
- Preprocessing inside a pipeline: median imputation for numbers, most-frequent imputation and one-hot encoding for `Student_Type`. Logistic regression also gets scaled features.
- Because only 30% of students are stressed, a model that always says "not stressed" would score about 70% accuracy and be useless. So the models use balanced class weights, and we judge them mainly on **recall, F1 and ROC-AUC**, not accuracy alone.

Models compared: Logistic Regression (baseline), Decision Tree, Random Forest, Gradient Boosting.

In [ ]:
features = ["Student_Type"] + num_cols + ["Month"]
X = clean[features]
y = clean["Stress_Level"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print(f"Train: {X_train.shape[0]:,} rows | Test: {X_test.shape[0]:,} rows")
print(f"Stressed share  train: {y_train.mean():.3f}  test: {y_test.mean():.3f}")

numeric_features = num_cols + ["Month"]
categorical_features = ["Student_Type"]

def make_preprocessor(scale):
    num_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num_steps), numeric_features),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
    ])

models = {
    "Logistic Regression": Pipeline([
        ("prep", make_preprocessor(scale=True)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]),
    "Decision Tree": Pipeline([
        ("prep", make_preprocessor(scale=False)),
        ("clf", DecisionTreeClassifier(max_depth=6, min_samples_leaf=50,
                                       class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "Random Forest": Pipeline([
        ("prep", make_preprocessor(scale=False)),
        ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=10,
                                       class_weight="balanced", n_jobs=-1,
                                       random_state=RANDOM_STATE)),
    ]),
    "Gradient Boosting": Pipeline([
        ("prep", make_preprocessor(scale=False)),
        ("clf", HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, max_depth=4,
                                               class_weight="balanced",
                                               random_state=RANDOM_STATE)),
    ]),
}

In [ ]:
# 5-fold cross-validation on the training set only
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, model in models.items():
    auc = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    f1 = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1)
    cv_rows.append({"Model": name,
                    "CV ROC-AUC": auc.mean(), "CV ROC-AUC std": auc.std(),
                    "CV F1": f1.mean(), "CV F1 std": f1.std()})
cv_results = pd.DataFrame(cv_rows).set_index("Model").round(4)
cv_results

## 5. Evaluation on the held-out test set

In [ ]:
rows, fitted, probs = [], {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    fitted[name], probs[name] = model, proba
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
    })
results = pd.DataFrame(rows).set_index("Model").round(4).sort_values("ROC-AUC", ascending=False)
results

In [ ]:
best_name = results.index[0]
best_model = fitted[best_name]
print("Best model by ROC-AUC:", best_name)
print()
print(classification_report(y_test, best_model.predict(X_test),
                            target_names=["Not stressed", "Stressed"], digits=3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_estimator(
    best_model, X_test, y_test, display_labels=["Not stressed", "Stressed"],
    cmap="Blues", values_format="d", ax=axes[0], colorbar=False)
axes[0].set_title(f"Confusion matrix: {best_name}")
axes[0].grid(False)

for name in results.index:
    RocCurveDisplay.from_predictions(y_test, probs[name], name=name, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[1].set_title("ROC curves")
axes[1].legend(loc="lower right", fontsize=9)

plt.tight_layout()
save(fig, "06_confusion_matrix_and_roc")
plt.show()

### Which factors drive stress?

Permutation importance shuffles one feature at a time on the test set and measures how much the ROC-AUC drops. A bigger drop means the model relies on that feature more.

In [ ]:
perm = permutation_importance(best_model, X_test, y_test, scoring="roc_auc",
                              n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.DataFrame({"feature": X_test.columns,
                    "importance": perm.importances_mean,
                    "std": perm.importances_std}).sort_values("importance")

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(imp["feature"].str.replace("_", " "), imp["importance"], xerr=imp["std"], color="#4C78A8")
ax.set_xlabel("Drop in ROC-AUC when shuffled")
ax.set_title(f"Permutation importance: {best_name}")
plt.tight_layout()
save(fig, "07_feature_importance")
plt.show()

imp.sort_values("importance", ascending=False).round(4).reset_index(drop=True)

In [ ]:
# Direction of each effect, from the logistic regression coefficients (features are standardized)
lr = fitted["Logistic Regression"]
names = lr.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(lr.named_steps["clf"].coef_[0], index=names)
coefs.index = [c.split("__")[1] for c in coefs.index]
coefs = coefs[numeric_features].sort_values()   # numeric features only; one-hot columns are not directly comparable

fig, ax = plt.subplots(figsize=(8, 5.5))
colors = ["#E45756" if v > 0 else "#4C78A8" for v in coefs.values]
ax.barh(coefs.index, coefs.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Logistic regression coefficients\n(red raises stress, blue lowers it)")
plt.tight_layout()
save(fig, "08_logistic_coefficients")
plt.show()

## 6. Conclusions

- **Stress is fairly predictable.** The best models reach a ROC-AUC of about 0.87 and catch roughly 76% of stressed students (recall) with about 80% overall accuracy. Gradient boosting, logistic regression and random forest perform almost identically, so the signal in this data is mostly simple and close to linear.
- **Exam pressure is the dominant driver by a wide margin.** Shuffling it drops the ROC-AUC by about 0.28, roughly seven times more than the next feature.
- **Study hours come second** and push stress up. **Sleep and family support protect** against stress, and **social media time** adds a little. Attendance and month matter very little.
- **Student type looks important in the raw rates but adds almost nothing to the model.** Working students are stressed most often (about 38%), college students 31% and school students 20%. But average study hours differ by type (about 6.1, 5.0 and 3.0 hours), and the model already gets that information from `Study_Hours`.
- **Class imbalance matters.** Only 30% of students are stressed, so accuracy alone would be misleading. Balanced class weights traded some precision (about 63%) for much better recall.

**Limitations**
- About 5% of every column was missing and was filled with the median; a few impossible values were removed.
- The data shows association, not cause. High exam pressure goes together with stress, but this data cannot prove it causes it.
- The stress label is binary, so it hides how severe the stress is.

**Possible extensions:** tune hyperparameters with a grid search, try other imputation methods, add interaction features (for example exam pressure x sleep), or turn the model into a small web app.

## 7. Try it on a new student

The trained model can score any student profile. Replace the numbers below with your own values.

In [ ]:
def predict_stress(student_type, sleep, study, social_media, attendance,
                   exam_pressure, family_support, month, model=best_model):
    row = pd.DataFrame([{
        "Student_Type": student_type, "Sleep_Hours": sleep, "Study_Hours": study,
        "Social_Media_Hours": social_media, "Attendance": attendance,
        "Exam_Pressure": exam_pressure, "Family_Support": family_support, "Month": month,
    }])[features]
    p = model.predict_proba(row)[0, 1]
    return p, ("Stressed" if p >= 0.5 else "Not stressed")

examples = {
    "High pressure, little sleep": ("college", 4.5, 8.0, 5.0, 70, 9, 3, 11),
    "Relaxed, well supported":     ("school", 8.0, 3.0, 2.0, 92, 3, 8, 6),
}
for label, args in examples.items():
    p, verdict = predict_stress(*args)
    print(f"{label:30s} -> {verdict} (probability of stress {p:.1%})")